In [1]:
import numpy as np
from scipy import optimize
from scipy.constants import mu_0, epsilon_0
from scipy import fftpack
from scipy import sparse
from scipy.special import factorial
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d, CubicSpline,splrep, BSpline
from scipy.sparse import csr_matrix, csc_matrix
import csv
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.animation as animation
from IPython.display import display, Math, Latex, Markdown
from scipy.linalg import lu_factor, lu_solve
from scipy import signal
import ipywidgets as widgets
import empymod
import discretize
import  os
import glob
import json
import pandas as pd
import time 


In [2]:
from masa_utils import sci_latex
from masa_utils import InducedPolarizationSimulation
from masa_utils import Pelton_res_f, DDR_f, DDC_f
from masa_utils import Optimization 

In [3]:
plt.style.use('00_video.mplstyle')


In [4]:
# --- helper to render LaTeX labels (robust method) ---
def math_label(tex, width="160px"):
    out = widgets.Output(layout=widgets.Layout(width=width))
    with out:
        display(Math(tex))
    return out
def sci_latex(v, prec=2):
    s = f"{v:.{prec}e}"          # e.g. '3.00e-03'
    mant, exp = s.split('e')
    exp = int(exp)

    if float(mant) == 0:
        return "0"

    if exp in [0, -1]:
        return f"{float(mant)*10**exp:.{prec}f}"
    else:
        return rf"{mant}\cdot 10^{{{exp}}}"


def fmt(v, prec=2, latex=False, wrap=False):
    # blank for None or empty string
    if v is None or v == "":
        return ""

    # if already string → return as-is
    if isinstance(v, str):
        return v

    try:
        s = sci_latex(v, prec=prec)

        if latex:
            if wrap:
                return rf"$${s}$$"   # display math
            else:
                return rf"${s}$"     # inline math
        else:
            return s

    except:
        return ""


Prepare SIP to fit using Pelton model

In [ ]:
# -----------------------------
# 2) Define your base values and sweep ranges
# -----------------------------
# Use your defaults here:
rho0_base = 0.3
eta_base  = 0.4
tau_base  = 1e-1
c_base    = 0.5

# Choose sweep ranges (edit these!)
tau_vals  = np.logspace(start=-4, stop=-1,base=10, num=121)
# c_vals    = np.linspace(start=0.3, stop=0.8, num = 101)

m_trues_1 = []
m_trues_2 = []
nmodel_cmb=[]

# sweep tau_rho
for t in tau_vals:
    m_trues_1.append(np.r_[np.log(rho0_base), eta_base, np.log(t), c_base])
for t in tau_vals[::-1]:  # reverse the list to go back to the start
    m_trues_1.append(np.r_[np.log(rho0_base), eta_base, np.log(t), c_base])

nmodel_cmb.append(len(m_trues_1))
# sweep c
for t in tau_vals:
    m_trues_2.append(np.r_[np.log(rho0_base), eta_base, np.log(t), c_base])
for t in tau_vals[::-1]:  # reverse the list to go back to the start
    m_trues_2.append(np.r_[np.log(rho0_base), eta_base, np.log(t), c_base])

nmodel_cmb.append(len(m_trues_2))
m_trues_cmb = []
m_trues_cmb.append(m_trues_1)
m_trues_cmb.append(m_trues_2)
print(nmodel_cmb)


[242, 242]


Prepare Widget

In [45]:
# Set Pelton for generating synthetic data
sim_obs_cmb=[]
sim_obs_cmb_ext=[]
sim_inv_cmb=[]
sim_inv_cmb_ext=[]
m_refs=[]
freq_mins = []
freq_maxs = []


freq_step_log = 0.1
taus_step_log = 0.5


freq_start_log = 0
freq_end_log = 4
nfreq= int((freq_end_log - freq_start_log) / freq_step_log) + 1
freq = np.logspace(freq_start_log, freq_end_log, nfreq)
ntau = int(round((freq_end_log - freq_start_log) / taus_step_log)) + 1
taus = np.logspace(-freq_end_log, -freq_start_log, ntau, base=10.0) / (2.0*np.pi)
freq_mins.append(freq.min())
freq_maxs.append(freq.max())
IP_model = Pelton_res_f(freq=freq)
sim_obs_cmb.append(InducedPolarizationSimulation(ip_model=IP_model, mode="sip"))
IP_model =DDR_f(freq=freq, taus=taus)
sim_inv_cmb.append(InducedPolarizationSimulation(ip_model=IP_model, mode="sip"))
m_ref = np.r_[np.log(rho0_base), np.zeros(ntau)]
m_refs.append(m_ref) 

freq_start_log = -1
freq_end_log = 5
nfreq= int((freq_end_log - freq_start_log) / freq_step_log) + 1
freq_ext = np.logspace(freq_start_log, freq_end_log, nfreq)
ntau = int(round((freq_end_log - freq_start_log) / taus_step_log)) + 1
taus_ext = np.logspace(-freq_end_log, -freq_start_log, ntau, base=10.0) / (2.0*np.pi)
IP_model = Pelton_res_f(freq=freq_ext)
sim_obs_cmb_ext.append(InducedPolarizationSimulation(ip_model=IP_model, mode="sip"))
IP_model =DDR_f(freq=freq_ext, taus=taus)
sim_inv_cmb_ext.append(InducedPolarizationSimulation(ip_model=IP_model, mode="sip"))
taus_xlim_min, taus_xlim_max = taus_ext.min()*0.9, taus_ext.max()*1.1
freq_xlim_min, freq_xlim_max = freq_ext.min()*0.9, freq_ext.max()*1.1


freq_start_log = 2
freq_end_log = 4
freq_step_log = 0.1
nfreq= int((freq_end_log - freq_start_log) / freq_step_log) + 1
freq = np.logspace(freq_start_log, freq_end_log, nfreq)
ntau = int(round((freq_end_log - freq_start_log) / taus_step_log)) + 1
taus = np.logspace(-freq_end_log, -freq_start_log, ntau, base=10.0) / (2.0*np.pi)
freq_mins.append(freq.min())
freq_maxs.append(freq.max())
IP_model = Pelton_res_f(freq=freq, chglim=0)
sim_obs_cmb.append(InducedPolarizationSimulation(ip_model=IP_model, mode="sip"))
IP_model =DDR_f(freq=freq, taus=taus)
sim_inv_cmb.append(InducedPolarizationSimulation(ip_model=IP_model, mode="sip"))
m_ref = np.r_[np.log(rho0_base), np.zeros(ntau)]
m_refs.append(m_ref) 

IP_model = Pelton_res_f(freq=freq_ext)
sim_obs_cmb_ext.append(InducedPolarizationSimulation(ip_model=IP_model, mode="sip"))
IP_model =DDR_f(freq=freq_ext, taus=taus)
sim_inv_cmb_ext.append(InducedPolarizationSimulation(ip_model=IP_model, mode="sip"))


In [22]:
nmodel_cmb

[242, 242]

In [23]:
relative_error = 0.02
noise_floor_ratio = 1e-3
niter = 100
stol=1e-6
coolingFactor = 2.0
coolingRate = 2
mu=1e-3
beta0_ratio = 1
alphas= 1.0
Ws_threshold=1e-3

In [24]:
dobs_cmb=[]
dobs_cmb_abs_max = []
dobs_cmb_abs_min= []
noise_floor_cmb=[]
mpreds_cmb = []
dpreds_cmb = []
model_prgs_cmb = []
data_prgs_cmb = []
error_prgs_cmb = []
for j in range(2):
    sim_obs= sim_obs_cmb[j]
    m_trues = m_trues_cmb[j]
    sim_inv = sim_inv_cmb[j]
    m_ref = m_refs[j]
    dobs_cmb_j = []
    dobs_cmb_abs_max_j = []
    dobs_cmb_abs_min_j = []
    noise_floor_cmb_j = []
    mpreds_cmb_j = []
    dpreds_cmb_j = []
    model_prgs_cmb_j = []
    data_prgs_cmb_j = []
    error_prgs_cmb_j = []
    for i in range(nmodel_cmb[j]):
        m_true = m_trues[i]
        dobs = sim_obs.dpred(m_true)
        f_abs = abs(sim_obs.ip_model.f(m_true))
        dobs_cmb_j.append(dobs)
        dobs_cmb_abs_max_j.append(np.max(f_abs))
        dobs_cmb_abs_min_j.append(np.min(f_abs))
        noise_floor =noise_floor_ratio * np.max(f_abs)
        noise_floor_cmb_j.append(noise_floor)
        opt = Optimization(sim=sim_inv, dobs=dobs, alphas=alphas, Ws_threshold=Ws_threshold)
        opt.get_Wd(ratio=relative_error, plateau=noise_floor)
        opt.get_Ws(smallness=np.ones(len(m_ref)))
        beta0 = opt.BetaEstimate_byEig(
        mvec=m_ref, update_Wsen=True, beta0_ratio=beta0_ratio)
        print(beta0)
        mpred = opt.GaussNewton(
        mvec_init=m_ref,niter=niter,beta0=beta0, update_Wsen=True,
        stol=stol,mu=mu,coolingRate=coolingRate, coolingFactor=coolingFactor
        )

        mpreds_cmb_j.append(mpred)
        dpreds_cmb_j.append(opt.dpred(mpred))
        error_prgs_cmb_j.append(opt.error_prg)
        model_prgs_cmb_j.append(opt.mvec_prg)
        data_prgs_cmb_j.append(opt.data_prg)

    noise_floor_cmb.append(noise_floor_cmb_j)
    dobs_cmb.append(dobs_cmb_j)
    dobs_cmb_abs_max.append(dobs_cmb_abs_max_j)
    dobs_cmb_abs_min.append(dobs_cmb_abs_min_j)
    mpreds_cmb.append(mpreds_cmb_j) 
    dpreds_cmb.append(dpreds_cmb_j)
    error_prgs_cmb.append(error_prgs_cmb_j)
    model_prgs_cmb.append(model_prgs_cmb_j)
    data_prgs_cmb.append(data_prgs_cmb_j)

40370928.65843591
  1, beta:4.0e+07, step:1.0e+00, g:5.0e+05, phid:4.8e+04, phim:9.3e-05, f:5.2e+04 
  2, beta:4.0e+07, step:6.2e-02, g:6.4e+03, phid:4.8e+04, phim:9.2e-05, f:5.2e+04 
  3, beta:2.0e+07, step:1.0e+00, g:2.1e+05, phid:3.8e+04, phim:2.8e-04, f:4.3e+04 
  4, beta:2.0e+07, step:6.2e-02, g:4.4e+03, phid:3.8e+04, phim:2.8e-04, f:4.3e+04 
  5, beta:1.0e+07, step:1.0e+00, g:1.9e+05, phid:2.5e+04, phim:7.4e-04, f:3.3e+04 
  6, beta:1.0e+07, step:1.2e-01, g:5.4e+03, phid:2.5e+04, phim:7.4e-04, f:3.3e+04 
  7, beta:5.0e+06, step:1.0e+00, g:1.5e+05, phid:1.3e+04, phim:1.6e-03, f:2.1e+04 
  8, beta:5.0e+06, step:1.0e+00, g:4.9e+03, phid:1.3e+04, phim:1.6e-03, f:2.1e+04 
  9, beta:2.5e+06, step:1.0e+00, g:1.1e+05, phid:5.6e+03, phim:2.6e-03, f:1.2e+04 
 10, beta:2.5e+06, step:1.0e+00, g:3.1e+03, phid:5.7e+03, phim:2.6e-03, f:1.2e+04 
 11, beta:1.3e+06, step:1.0e+00, g:7.2e+04, phid:2.0e+03, phim:3.6e-03, f:6.5e+03 
 12, beta:1.3e+06, step:9.5e-07, g:1.4e+03, phid:2.0e+03, phim:3.6e-0

In [26]:
phid_star_ratio = 0.5
models_rec_phid_cmb = []

for j in range(2):
    model_prgs_j = model_prgs_cmb[j]
    dpreds_cmb_j = dpreds_cmb[j]
    data_prgs_cmb_j = data_prgs_cmb[j]
    error_prgs_cmb_j = error_prgs_cmb[j]
    models_rec_phid_cmb_j = []
    for i in range(nmodel_cmb[j]):
        phid_star = nfreq*2
        model_prg= np.array(model_prgs_j[i])
        data_prg = np.array(dpreds_cmb_j[i])
        error_prg = error_prgs_cmb_j[i]
        phid_prg = np.array(error_prg)[:,1]
        ind = phid_prg < phid_star*phid_star_ratio
        if np.sum(ind) == 0:
            ind = phid_prg == np.min(phid_prg)
            print(f" range {i}:No phid star found, take min phid")
        model = model_prg[ind][0]
        models_rec_phid_cmb_j.append(model)
        rho0_sip = np.exp(model[0])
        eta_sip = model[1:].sum()
        print(f"Model {i}: rho0: {rho0_sip:.2e}, eta: {eta_sip:.3f}")
    models_rec_phid_cmb.append(models_rec_phid_cmb_j)


Model 0: rho0: 2.98e-01, eta: 0.370
Model 1: rho0: 2.98e-01, eta: 0.370
Model 2: rho0: 2.98e-01, eta: 0.370
Model 3: rho0: 2.97e-01, eta: 0.371
Model 4: rho0: 2.97e-01, eta: 0.371
Model 5: rho0: 2.97e-01, eta: 0.371
Model 6: rho0: 2.97e-01, eta: 0.371
Model 7: rho0: 2.97e-01, eta: 0.371
Model 8: rho0: 2.97e-01, eta: 0.372
Model 9: rho0: 2.97e-01, eta: 0.372
Model 10: rho0: 2.97e-01, eta: 0.372
Model 11: rho0: 2.97e-01, eta: 0.372
Model 12: rho0: 2.97e-01, eta: 0.372
Model 13: rho0: 2.97e-01, eta: 0.372
Model 14: rho0: 2.97e-01, eta: 0.372
Model 15: rho0: 2.96e-01, eta: 0.373
Model 16: rho0: 2.96e-01, eta: 0.373
Model 17: rho0: 2.96e-01, eta: 0.373
Model 18: rho0: 2.96e-01, eta: 0.373
Model 19: rho0: 2.96e-01, eta: 0.373
Model 20: rho0: 2.96e-01, eta: 0.373
Model 21: rho0: 2.96e-01, eta: 0.373
Model 22: rho0: 2.96e-01, eta: 0.373
Model 23: rho0: 2.96e-01, eta: 0.373
Model 24: rho0: 2.96e-01, eta: 0.373
Model 25: rho0: 2.95e-01, eta: 0.373
Model 26: rho0: 2.95e-01, eta: 0.373
Model 27: r

# Plot time

In [27]:

print("usetex =", mpl.rcParams["text.usetex"])

usetex = False


In [36]:
#  Widget for the main function
def plot_DD_widget(
         j,i, widget=True,
         eta_hline_threshold=0.10, axsip=None,axetas=None):

    # Create the plot
    if widget:
        fig, ax = plt.subplots(3, 1, gridspec_kw={"height_ratios":[1.8,1,1]})
        fig.subplots_adjust(right=0.75)
        axsip = [ax[0], ax[1]]
        axetas = ax[2]
    else:
        assert axsip is not None
        assert axetas is not None
        axsip[0].clear()
        axsip[1].clear()
        axetas.clear()
    sim_obs = sim_obs_cmb[j]
    sim_inv = sim_inv_cmb[j]
    sim_obs_ext = sim_obs_cmb_ext[j]
    sim_inv_ext = sim_inv_cmb_ext[j]
    m_true = m_trues_cmb[j][i]
    mpred = models_rec_phid_cmb[j][i]
    freq_min = freq_mins[j]
    freq_max = freq_maxs[j]

    axsip = sim_inv_ext.plot_sip_model(mpred, ax=axsip, color='C0', linestyle="-.", alpha=0.7 )
    axsip = sim_inv.plot_sip_model(mpred, ax=axsip, color='C0')
    axetas = sim_inv.ip_model.plot_etas(mpred,
            ax=axetas, color="C0",
            label="Est",
            linestyle="-", marker='o', markersize=4.0
            )

    axsip = sim_obs_ext.plot_sip_model(m_true, ax=axsip, color='C3', linestyle=':', alpha=0.7)
    axsip = sim_obs.plot_sip_model(m_true, ax=axsip, linestyle=':', color='C3')
    
    abs_max= dobs_cmb_abs_max[j][i]
    abs_min = dobs_cmb_abs_min[j][i]
    rho0_DD = np.exp(mpred[0])
    rho0_pel = np.exp(m_true[0])
    eta_DD = mpred[1:].sum()
    eta_pel =m_true[1]
    tau_pel = np.exp(m_true[2])
    cc_pel = m_true[3]

    axsip[0].axhline(y=rho0_DD, color='C0', linestyle='--')
    axsip[0].axhline(y=rho0_pel, color='C3', linestyle=':')
    axsip[0].axhline(y=rho0_DD*(1-eta_DD), color='C0', linestyle='--')
    axsip[0].axhline(y=rho0_pel*(1-eta_pel), color='C3', linestyle=':')

    text = (
        f"DD ρ₀ : {rho0_DD:.2f}\n"
        f"|max(ρ)| : {abs_max:.2f}\n"
        f"PEL ρ₀ : {rho0_pel:.2f}\n\n"
        f"DD ρ₀(1-Σηₖ) : {rho0_DD*(1-eta_DD):.2f}\n"
        f"|min(ρ)| : {abs_min:.2f}\n"
        f"PEL ρ₀(1-η) : {rho0_pel*(1-eta_pel):.2f}"
    )
    axsip[0].text(
        1.02, 1.0,
        text,
        transform=axsip[0].transAxes,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9)
    )
    for a in axsip:
        a.axvline(x=freq_min, color='k', linestyle=':')
        a.axvline(x=freq_max, color='k', linestyle='-.')
        a.set_xlim(left=freq_xlim_max, right=freq_xlim_min)
        a.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
    axetas.set_xlim(left=taus_xlim_min, right=taus_xlim_max)

    axsip[1].set_ylim([0, -8])
    m_true_text = r"m_true\n" + r"$\rho_0$" + f": {rho0_pel: .2f} \n " + r"$\eta$" +f": {eta_pel: .2f} \n"  
    m_true_text += r"$\tau_\rho$"+ f": ${fmt(tau_pel, prec=1)}$ \n" +r"$C$" + f": {cc_pel: .2f}"

    axsip[1].text(
        1.02, 1.0,
        m_true_text,
        transform=axsip[1].transAxes,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
    )

    m_pred_text = r"m_pred\n" + r"$\rho_0$" + f": {rho0_DD: .2f} \n " + r"$\eta$" +f": {eta_DD: .2f} \n"
    axetas.text(
        1.02, 1.0,
        m_pred_text,
        transform=axetas.transAxes,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
    )
    axetas.set_ylim([0, .2])
    axsip[0].set_xlabel("")
    if widget:
        return fig, axsip, axetas

In [ ]:
i_slider = widgets.IntSlider(
    min=0, max=nmodel_cmb[0]-1, step=1, value=0,
    description=""
)
j_slider = widgets.IntSlider(
    min=0, max=1, step=1, value=0,
    description=""
)

i_row = widgets.HBox([
    widgets.Label("Model index", layout=widgets.Layout(width="180px")),
    i_slider
])

j_row = widgets.HBox([
    widgets.Label("F-band index", layout=widgets.Layout(width="180px")),
    j_slider
])
i_slider.continuous_update = False
j_slider.continuous_update = False

In [38]:
plt.style.use('00_video.mplstyle')

In [41]:
# --- controls layout ---
controls = widgets.VBox([i_row, j_row])
# --- connect UI to function (NO duplicate sliders) ---
out = widgets.interactive_output(
    plot_DD_widget,
    {
        "i": i_slider,
        "j": j_slider,
    },
)

# --- display everything ---
display(controls, out)

Output()

In [44]:
# -----------------------------
# video writer (no PNGs)
# -----------------------------
fps = 60
dpi = 150
writer = animation.FFMpegWriter(fps=fps, metadata={"artist": "matplotlib"})
out_mp4_cmb= [
    "../figures/38_DD_animation_band1.mp4",
    "../figures/38_DD_animation_band2.mp4"
]
for j in range(2):
    i=0
    fig, axsip, axetas = plot_DD_widget(j=j,i=i, widget=True)
    out_mp4 = out_mp4_cmb[j]
    with writer.saving(fig, out_mp4, dpi=dpi):
        for i in range(1, nmodel_cmb[j]):
            # draw_pelton_on_axes(axsip, axui, rho0, eta, tau, c)
            plot_DD_widget(i=i,j=j,widget=False, axsip=axsip, axetas=axetas)
            fig.subplots_adjust(right=0.78)
            fig.canvas.draw()     # ensure the canvas is updated
            writer.grab_frame()   # grab this frame from *this same fig*

    plt.close(fig)
    print(f"Saved animation: {out_mp4}")

C:\Users\81805\AppData\Local\Temp\ipykernel_19344\1500687506.py:72: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  a.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
C:\Users\81805\AppData\Local\Temp\ipykernel_19344\1500687506.py:72: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  a.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))


Saved animation: ../figures/38_DD_animation_band1.mp4


C:\Users\81805\AppData\Local\Temp\ipykernel_19344\1500687506.py:72: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  a.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))


Saved animation: ../figures/38_DD_animation_band2.mp4
